# ShopDesk, Section 3 Lab 2: Parallel Subagents and Adaptive Decomposition

A beginner-friendly notebook that dispatches several ShopDesk subagents **in parallel** in a
single coordinator turn, contrasts a **fixed pipeline** with **adaptive decomposition**, and
**scores** the two strategies. Built on the **Claude Agent SDK**, running **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

At end of day ShopDesk has several orders to triage. Handling them one after another is slow,
and forcing every request through the same fixed steps does needless work (refunding an order
nobody asked to refund). Two ideas fix this: **fan out** independent work so subagents run at
once, and **decompose adaptively** so each request gets only the steps it needs.

The question this lab answers: **how do you run subagents in parallel in one turn, and how do
you tell whether adaptive decomposition actually beats a fixed pipeline?**

## Objectives

- Dispatch **multiple subagents in a single response** so independent work runs in parallel.
- Implement an **adaptive decomposition** that plans only the needed subtasks, versus a fixed
  **prompt-chaining** pipeline.
- **Evaluate** output quality across the two strategies with a small rubric.
- Note where a **forked session** helps you explore an alternate decomposition safely.

## What you'll observe

- The coordinator issues several Agent calls in one turn; the narrator flags the parallel
  dispatch.
- The fixed pipeline over-processes some requests and misses parts of others; the adaptive
  plan matches the request far more closely.
- The scorer prints a quality number per strategy, and adaptive wins on the mixed set.

## How to run

Run top to bottom. The planners and the evaluation are pure Python and run anywhere. The
parallel-dispatch cell calls Claude, so paste a real key into **Setup 2/3** and re-run from
the top; otherwise it skips. **Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. The **Agent SDK** runs the parallel coordinator;
the base SDK and dotenv handle the key. The Agent SDK also needs Node.js 18+, which cannot be
pip-installed.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` so the
async coordinator can be called like a normal function.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import json                                     # build and print planning payloads
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the coordinator will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **ShopDesk world**: a few orders to triage. The
coordinator fans out over these, and the planners below reason about them.

In [ ]:
# ===== SETUP 3/3 - the shared ShopDesk data =====
ORDERS = {                                       # the batch to triage
    "A1": {"status": 2, "refundable": True},     #   shipped
    "A3": {"status": 1, "refundable": True},     #   processing, late
    "A4": {"status": 2, "refundable": True},     #   shipped, defective report
}
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}   # status code -> human word
print("orders to triage:", list(ORDERS))         # quick sanity check

**This cell:** the **narrator** `stream_run()`. It runs one coordinator `query()` and,
crucially, reports when **more than one subagent is invoked in a single turn**, which is what
parallel dispatch looks like. Subagent calls arrive through the Agent tool (older name:
Task).

In [ ]:
# ===== stream one run and flag parallel dispatch =====
from claude_agent_sdk import (                     # the Agent SDK pieces we use:
    query, ClaudeAgentOptions, AgentDefinition,    #   run, options, subagent spec
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,   # message + block types
)

async def stream_run(options, prompt):             # run query() and narrate the delegations
    print("USER:", prompt)                         #   echo the request
    answer = ""                                    #   keep the final aggregated text
    async for message in query(prompt=prompt, options=options):   # stream every message
        if isinstance(message, AssistantMessage):  #     the coordinator (or a subagent) spoke
            subs = [b for b in message.content      #       subagent invocations THIS turn
                    if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task")]
            if len(subs) > 1:                       #       several in one turn = parallel
                print(f"  PARALLEL: {len(subs)} subagents dispatched in one turn")
            for b in subs:                          #       list who got the work
                who = b.input.get("subagent_type", "?") if isinstance(b.input, dict) else "?"
                print("    -> delegate to:", who)
            for b in message.content:               #       keep the latest coordinator text
                if isinstance(b, TextBlock):
                    answer = b.text
        elif isinstance(message, ResultMessage):    #     the whole run finished
            print("  (run complete)")
    print("ANSWER:", answer)                        #   the combined answer
    return answer

---

### 🎯 Lab objective - fan out, adapt, and measure

**What you build:** a coordinator that dispatches one subagent per order in a single turn, two
decomposition strategies (fixed and adaptive), and a scorer that compares them.

**Why it helps you build real solutions:** parallel dispatch turns a slow serial triage into
one concurrent pass, and adaptive decomposition stops you paying for steps a request never
needed. Measuring the two is how you justify the choice.

**How you'll see it:** the narrator flags parallel dispatch, and the scorer shows adaptive
beating the fixed pipeline on a mixed set.

**This cell:** a coordinator built for **parallel triage**. It registers one focused
`triage-agent` and is told to dispatch one per order **in a single step** so they run
concurrently. Independent work (different orders) is exactly what should fan out.

In [ ]:
# ===== a coordinator that fans out one subagent per order =====
triage_agent = AgentDefinition(                    # one focused spoke, reused per order
    description="Triage a single order: summarise its status and the right next action.",
    prompt="You triage ONE order. State its status and the single best next action. Be brief.",
    tools=["Read"], model="sonnet")

PARALLEL = ClaudeAgentOptions(                     # coordinator settings for fan-out
    model=MODEL,
    system_prompt=("You are the ShopDesk triage coordinator. When several orders are "
                   "mentioned, dispatch one triage-agent per order in a SINGLE step so they "
                   "run in parallel, then combine their summaries into one list."),
    agents={"triage-agent": triage_agent},          #   register the single spoke
    allowed_tools=["Agent", "Read"])                #   "Agent" unlocks delegation
print("parallel coordinator ready")

**This cell:** runs the parallel coordinator on a batch of three orders. Watch the
narrator: if the coordinator dispatches all three triage-agents in one turn, it prints the
`PARALLEL` line. (Model behaviour is not perfectly deterministic, so treat it as an
observation.)

In [ ]:
# ===== run the parallel triage =====
batch = "Triage orders A1, A3, and A4. Handle each order independently."   # three independent jobs
if RUN_LIVE:                                      # needs a real key (and Node.js 18+)
    run_async(lambda: stream_run(PARALLEL, batch))
else:
    print("[skipped] expected: three triage-agent calls in ONE turn (parallel), then a")
    print("          combined three-line summary.")

**This cell:** the **fixed pipeline** (prompt chaining), as pure Python. It always plans
the same steps in the same order for every request, no matter what was asked. It runs offline
and is our baseline to beat.

In [ ]:
# ===== strategy A: the fixed pipeline (always the same steps) =====
def fixed_plan(request):                           # request -> a FIXED set of subtasks
    return {"shipping", "refund"}                  #   always status + refund, regardless of the ask
print("fixed_plan always returns:", fixed_plan("anything"))

**This cell:** the **adaptive** strategy. It reads the request and plans only the
subtasks the request implies (shipping, refund, escalation), so the work matches the ask. It
too runs offline.

In [ ]:
# ===== strategy B: adaptive decomposition (plan only what is needed) =====
def classify(request):                             # request text -> the intents present
    t = request.lower()                            #   normalise
    got = set()                                    #   collect intents
    if any(w in t for w in ["where", "status", "track", "shipped", "arrive", "late", "delivery"]):
        got.add("shipping")
    if any(w in t for w in ["refund", "money back", "return"]):
        got.add("refund")
    if any(w in t for w in ["furious", "angry", "upset", "unacceptable", "complaint", "defective"]):
        got.add("escalation")
    return got or {"shipping"}                      #   default to shipping if nothing matched

def adaptive_plan(request):                        # request -> only the needed subtasks
    return classify(request)                        #   the plan IS the detected intents
print("adaptive_plan('Where is A1?') ->", adaptive_plan("Where is A1?"))

**This cell:** the **evaluation set**: a handful of requests, each hand-labelled with
the subtasks it truly needs. This ground truth is what both strategies are graded against.

In [ ]:
# ===== the labelled requests we grade against =====
CASES = [                                          # (request, the subtasks it truly needs)
    ("Where is order A1?",                         {"shipping"}),
    ("Please refund order A2.",                    {"refund"}),
    ("Where is A1, and can I refund A2?",          {"shipping", "refund"}),
    ("A3 is late and I am furious.",               {"shipping", "escalation"}),
    ("A4 arrived defective, I want my money back.",{"refund", "escalation"}),
]
print("evaluation cases:", len(CASES))

**This cell:** the **scorer**. For each case it compares a strategy's plan to the
labelled truth and counts **spurious** steps (planned but not needed) and **missing** steps
(needed but not planned). A case is exact only when both are zero; the score is the fraction of
exact cases.

In [ ]:
# ===== grade a strategy across the evaluation set =====
def evaluate(plan_fn, name):                       # plan_fn: request -> set of subtasks
    exact = 0                                       #   count perfectly handled cases
    for request, needed in CASES:                   #   walk every labelled case
        plan = plan_fn(request)                     #     what the strategy would do
        spurious = plan - needed                    #     steps it added but should not
        missing = needed - plan                     #     steps it skipped but should not
        ok = not spurious and not missing           #     exact only if both are empty
        exact += ok                                 #     tally
        print(f"  {name:8} {request[:34]!r:36} plan={sorted(plan)} "
              f"spurious={sorted(spurious)} missing={sorted(missing)}")
    print(f"  {name} quality: {exact}/{len(CASES)} exact\n")   # the headline number
    return exact

print("Strategy A - fixed pipeline:")
score_fixed = evaluate(fixed_plan, "fixed")
print("Strategy B - adaptive decomposition:")
score_adaptive = evaluate(adaptive_plan, "adaptive")
print("winner:", "adaptive" if score_adaptive > score_fixed else "fixed")

**This cell:** a note on **forked sessions** for exploration. When you are unsure which
decomposition is right, fork the coordinator's session (from Section 2) and try an alternate
plan on the branch, leaving the original intact. The options below show the shape; capture a
`session_id` first, exactly as in Section 2 Lab 2.

In [ ]:
# ===== forked session for parallel exploration (shape only) =====
# Given a captured session_id from a prior coordinator run:
#   EXPLORE = ClaudeAgentOptions(model=MODEL, resume=session_id, fork_session=True)
#   run_async(lambda: stream_run(EXPLORE, "Try decomposing this a different way: ..."))
# The fork copies the history and diverges; the original session is left untouched,
# so you can compare two decomposition strategies as independent branches.
print("fork pattern: ClaudeAgentOptions(resume=<id>, fork_session=True)  # explore safely")

| anti-pattern | what to do instead |
|---|---|
| triage orders one after another | dispatch one subagent per order in a single turn |
| force every request through fixed steps | decompose adaptively to only the needed subtasks |
| pick a strategy by gut feel | score strategies against labelled cases |
| branch by editing the live session | fork the session so the original stays intact |

**Lesson:** fan out independent work so subagents run in one parallel turn, and let the
request decide its own decomposition instead of a rigid pipeline. Fixed prompt-chaining suits
truly predictable tasks; adaptive decomposition wins whenever requests vary, and a quick
rubric proves which one fits your traffic.

---

## Recap - parallelism, adaptivity, and evidence

| Idea | In this lab | Course topic |
|---|---|---|
| Parallel dispatch | one triage-agent per order in a single turn | multiple Agent calls in one response |
| Fixed pipeline | `fixed_plan` always the same steps | prompt chaining for predictable tasks |
| Adaptive decomposition | `adaptive_plan` from `classify` | dynamic decomposition for varied tasks |
| Evaluation | spurious + missing steps per case | comparing output quality across strategies |
| Forked exploration | `resume=<id>, fork_session=True` | fork-based exploration |

One principle to carry forward: **parallelise what is independent, decompose to fit the
request, and measure before you commit to a strategy.** To run the live cells, paste a real
key into **Setup 2/3** and re-run from the top. Then try it: add a case that needs all three
subtasks and watch the fixed pipeline fall further behind.